# IFJC4002 | Practical Artificial Intelligence with Python
## Case 2: Used Car Resale Price Prediction (Regression)
**Faculty of Intelligent Computing, Universiti Malaysia Perlis (UniMAP)**  
**Course Coordinator:** Ir. Dr. Marni Azira Markom  
**Group 1**

---
### Project Objectives:
1. **Task 1 (CO1):** Problem Definition and Exploratory Data Analysis (EDA).
2. **Task 2 (CO2):** Data Preprocessing Pipeline, Baseline Model, Algorithm Comparison (Linear Regression, Ridge, Random Forest, Gradient Boosting), and Metric Evaluation (MAE, RMSE, R²).
3. **Task 3 (CO3):** Pipeline Export, Model Verification, and Streamlit Integration.

In [ ]:
# 1. Setup and Library Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

In [ ]:
# 2. Load Dataset
# When running on Google Colab, you can mount your Drive or upload the Excel file directly
data_path = 'data/raw/02_Used_Car_Price.xlsx'
if not os.path.exists(data_path):
    data_path = '../data/raw/02_Used_Car_Price.xlsx'
if not os.path.exists(data_path):
    data_path = '02_Used_Car_Price.xlsx'

df = pd.read_excel(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

### Task 1: Exploratory Data Analysis (CO1)
- Check data types and missing values
- Descriptive statistics
- Feature correlation with target `resale_price_rm`

In [ ]:
# Missing Value Audit
missing = df.isnull().sum()
print("Missing Values per Column:")
print(missing[missing > 0])

# Numerical Summary
df.describe().round(2)

In [ ]:
# Visualizing Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['resale_price_rm'], kde=True, color='royalblue', ax=axes[0])
axes[0].set_title('Resale Price Distribution (RM)')
axes[0].set_xlabel('Resale Price (RM)')

sns.boxplot(x='market_segment', y='resale_price_rm', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Resale Price by Market Segment')
axes[1].set_ylabel('Resale Price (RM)')
plt.tight_layout()
plt.show()

### Task 2: Data Preprocessing & Model Evaluation (CO2)
- Build preprocessing pipeline (no data leakage: fitted on training set only)
- Evaluate Baseline (Mean Predictor)
- Train & compare 4 algorithms: Linear Regression, Ridge, Random Forest, Gradient Boosting
- Report MAE, RMSE, and R²

In [ ]:
# Exclude record_id and split into X and y
features = [
    'vehicle_age_years', 'mileage_km', 'engine_size_l',
    'fuel_type', 'transmission', 'market_segment',
    'condition_rating', 'previous_owners'
]
X = df[features]
y = df['resale_price_rm']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

# Preprocessor Pipeline
num_cols = ['vehicle_age_years', 'mileage_km', 'engine_size_l', 'condition_rating', 'previous_owners']
cat_cols = ['fuel_type', 'transmission', 'market_segment']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), cat_cols)
])
print("Preprocessor built successfully!")

In [ ]:
# Baseline Model (Mean Predictor)
y_base_pred = np.full_like(y_test, fill_value=y_train.mean())
mae_base = mean_absolute_error(y_test, y_base_pred)
rmse_base = root_mean_squared_error(y_test, y_base_pred)
r2_base = r2_score(y_test, y_base_pred)

print("--- Baseline Model (Mean Predictor) ---")
print(f"MAE : RM {mae_base:,.2f}")
print(f"RMSE: RM {rmse_base:,.2f}")
print(f"R²  : {r2_base:.4f}")

In [ ]:
# Model Training & Comparison
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
trained_pipelines = {}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2').mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        '5-Fold CV R²': round(cv_r2, 4),
        'Test MAE (RM)': round(mae, 2),
        'Test RMSE (RM)': round(rmse, 2),
        'Test R²': round(r2, 4)
    })
    trained_pipelines[name] = pipe

summary_df = pd.DataFrame(results)
display(summary_df)

### Task 3: Export Model & Test Prediction (CO3)
- Export best performing pipeline to `.joblib`
- Reload and verify end-to-end inference

In [ ]:
# Export Best Pipeline
best_model_name = summary_df.sort_values(by='Test R²', ascending=False).iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]
print(f"Best Model Selected: {best_model_name}")

os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/car_price_pipeline.joblib')
print("Exported pipeline to models/car_price_pipeline.joblib")

# Reload and test prediction
reloaded = joblib.load('models/car_price_pipeline.joblib')
sample_input = pd.DataFrame([{
    'vehicle_age_years': 4.0,
    'mileage_km': 50000.0,
    'engine_size_l': 2.0,
    'fuel_type': 'Petrol',
    'transmission': 'Automatic',
    'market_segment': 'Mid_Range',
    'condition_rating': 4,
    'previous_owners': 1
}])
pred_val = reloaded.predict(sample_input)[0]
print(f"Verified Test Prediction: RM {pred_val:,.2f}")